In [0]:
%sql
CREATE OR REPLACE VIEW cobranzas.semantic.vw_target_recuperacion_promesas_gestor AS

with uniques as (
    select distinct cuenta_id, monto_mensual_l
    from cobranzas.gold.fact_promesas
    where month(fecha_de_pago) = month(current_date)
      and year(fecha_de_pago) = year(current_date)
),

recuperado as (
    select distinct cuenta_id, monto_mensual_l
    from cobranzas.gold.fact_promesas
    where month(fecha_de_pago) = month(current_date)
    and year(fecha_de_pago) = year(current_date)
    and estado_promesa_id = 4
)

select
    r.gestor as gestor,
    sum(u.monto_mensual_l) as monto_mensual_l,
    sum(re.monto_mensual_l) as monto_recuperado
from uniques u
inner join cobranzas.gold.fact_cuenta c
    on u.cuenta_id = c.cuenta_id
inner join cobranzas.gold.dim_responsables_cuenta r
    on c.responsables_cuenta_id = r.responsables_cuenta_id
left join recuperado re
    on u.cuenta_id = re.cuenta_id
group by r.gestor